In [9]:
import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')  # Disable GPU temporarily to reconfigure
gpus = tf.config.list_physical_devices('GPU')

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
#tutorial https://www.kaggle.com/code/prashant111/random-forest-classifier-tutorial
data = "./features-relative.csv"
df = pd.read_csv(data)
patient_id = "BA0803901"

df_test = df[df['file'].str.contains(patient_id)]

# Train set: all other patients
df_train = df[~df['file'].str.contains(patient_id)]

X = df.drop(['Label', 'Start', 'End', 'file', 'Start Sample', 'End Sample'], axis=1)
X_train = df_train.drop(['Label', 'Start', 'End', 'file', 'Start Sample', 'End Sample'], axis=1)
X_test = df_test.drop(['Label', 'Start', 'End', 'file', 'Start Sample', 'End Sample'], axis=1)

# Labels
y_train = df_train['Label']
y_test = df_test['Label']

# Encode labels
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)  # only transform, don't fit again

2025-12-03 13:19:11.068531: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-03 13:19:11.133452: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Load data
data = "./features-relative.csv"
df = pd.read_csv(data)

# Feature matrix X (drop non-feature columns)
X = df.drop(['Label', 'Start', 'End', 'file', 'Start Sample', 'End Sample'], axis=1)

# Labels
y = df['Label']

# Encode labels
le = LabelEncoder()
y = le.fit_transform(y)

# Standard train/test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


Train size: (2811, 120)
Test size: (703, 120)


In [11]:

from sklearn.ensemble import RandomForestClassifier


rfc = RandomForestClassifier(random_state=0, n_estimators=300, class_weight="balanced")

rfc.fit(X_train, y_train)

y_pred = rfc.predict(X_test)

from sklearn.metrics import accuracy_score

print('Model accuracy score with 100 decision-trees : {0:0.4f}'. format(accuracy_score(y_test, y_pred)))

Model accuracy score with 100 decision-trees : 0.8592


In [17]:
from sklearn.metrics import confusion_matrix, classification_report

conf_matrix = confusion_matrix(y_test, y_pred)
print(conf_matrix)

# Classification Report
class_report = classification_report(y_test, y_pred)
print("Classification Report:")
print(class_report)

[[  0  94   1]
 [  0 366   4]
 [  0   0 238]]
Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        95
           1       0.80      0.99      0.88       370
           2       0.98      1.00      0.99       238

    accuracy                           0.86       703
   macro avg       0.59      0.66      0.62       703
weighted avg       0.75      0.86      0.80       703



/home/joaquinm/EMG-Thesis/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/joaquinm/EMG-Thesis/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/joaquinm/EMG-Thesis/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize

In [13]:
feature_scores = pd.Series(rfc.feature_importances_, index=X_train.columns).sort_values(ascending=False)

print(feature_scores)

NotFittedError: This RandomForestClassifier instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

file
.\Spontanaktivität\filtered_BP0803902_segment_1.wav     544
.\Spontanaktivität\filtered_BA0803901_segment_3.wav     423
.\Spontanaktivität\filtered_LI06056410_segment_1.wav    403
.\Spontanaktivität\filtered_HC280963_segment_1.wav      369
.\Spontanaktivität\filtered_RN181281_segment_3.wav      318
.\Spontanaktivität\filtered_BA0803901_segment_2.wav     247
.\Spontanaktivität\filtered_LI0605644_segment_1.wav     247
.\Spontanaktivität\filtered_HC280963_segment_2.wav      237
.\Spontanaktivität\filtered_LI0605645_segment_1.wav     194
.\Spontanaktivität\filtered_LI0605645_segment_4.wav     192
.\Spontanaktivität\filtered_LI0605645_segment_3.wav     175
.\Spontanaktivität\filtered_QD161095_segment_5.wav      170
.\Spontanaktivität\filtered_RN181281_segment_2.wav      167
.\Spontanaktivität\filtered_LI0605648_segment_2.wav     162
.\Spontanaktivität\filtered_BA0803901_segment_1.wav     157
.\Spontanaktivität\filtered_RN181281_segment_1.wav      156
.\Spontanaktivität\filtered_LI06056

Accuracy with normal split 0.89 accuracy
with patient split 0.78

In [ ]:
# Diagnostic: show label distribution and confusion matrix (robust)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# Print label counts in the test set
unique, counts = np.unique(y_test, return_counts=True)
print("y_test unique numeric labels:", dict(zip(unique, counts)))

# If LabelEncoder `le` exists, show the mapped class names, otherwise use numeric labels
try:
    class_names = [str(c) for c in le.classes_]
    print("LabelEncoder classes:", class_names)
except Exception:
    class_names = [str(int(u)) for u in unique]
    print("LabelEncoder not available — using numeric labels as names:", class_names)

# Make sure y_pred exists; if not, compute it from the trained model/rfc
try:
    y_pred
except NameError:
    print("y_pred not found — running prediction now")
    y_pred = rfc.predict(X_test)

# Build confusion matrix using all labels seen in y_test and y_pred
all_labels = np.unique(np.concatenate([y_test, y_pred]))
conf = confusion_matrix(y_test, y_pred, labels=all_labels)
print("Confusion matrix (rows=true, cols=pred):\n", conf)

# Safe mapping of numeric labels to names for display
display_labels = []
for lab in all_labels:
    try:
        # if le exists and supports inverse_transform
        display_labels.append(str(le.inverse_transform([lab])[0]))
    except Exception:
        display_labels.append(str(int(lab)))

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=conf, display_labels=display_labels)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

# Print detailed classification report using the labels we used
print('\nClassification report:')
print(classification_report(y_test, y_pred, labels=all_labels, target_names=display_labels, zero_division=0))
